# 00 · Setup and smoke test

Runs locally (VS Code, `bpp` conda env) **or** on Google Colab.

1. Detects where it is running.
2. On Colab: mounts Google Drive, clones the repo, installs it.
3. Runs the tests and the synthetic demo.
4. Shows a few outputs so you can see what each Phase 1–2 step produces.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
print('Running on Colab' if IN_COLAB else 'Running locally')

## Colab only: Drive + clone + install
Replace the repo URL. For a private repo, store a read-only GitHub token in Colab **Secrets** as `GITHUB_TOKEN`.

In [ ]:
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = 'github.com/<your-account>/<your-repo>.git'   # <-- change me
    try:
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@{REPO}'
    except Exception:
        url = f'https://{REPO}'
    if not os.path.exists('/content/bpp'):
        !git clone {url} /content/bpp
    %cd /content/bpp
    !pip install -q -r requirements.txt && pip install -q -e .
    !apt-get -qq install -y tesseract-ocr > /dev/null
    os.environ['BPP_DATA_DIR'] = '/content/drive/MyDrive/BPP-data'   # shared data folder

## Local only: go to the project root

In [ ]:
if not IN_COLAB:
    from pathlib import Path
    root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    os.chdir(root)
    print('project root:', root)

## Tests

In [ ]:
!python -m pytest -q

## Synthetic demo of Phases 1–2
Invented companies only; writes to `data_demo/`.

In [ ]:
!bpp demo --out data_demo

## Look at the outputs

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 60)
print('Name matches (note the needs_review row)')
display(pd.read_csv('data_demo/interim/ibbi_listed_matches.csv')[['corporate_debtor','matched_name','score','match_status']])
print('Cohort')
display(pd.read_csv('data_demo/processed/cohort.csv')[['pair_id','company_name','role','reference_date','total_assets_ref_fy','match_quality']])

In [ ]:
lab = pd.read_csv('data_demo/processed/documents_labeled.csv')
display(lab[['doc_id','role','fy','pub_date','months_before_reference','horizon','exclude_reason','audit_opinion']])

In [ ]:
import json
d = json.load(open('data_demo/interim/sections/BSE900101_FY2018.json'))
for name, sec in d['sections'].items():
    print(f"{name:<28}", (f"pages {sec.get('start_page','-')}-{sec.get('end_page','-')}, {sec['n_chars']} chars" if sec else 'not found'))
print('\naudit opinion:', d['audit_opinion'])
print('\nGoing-concern paragraph:\n', d['sections']['going_concern']['text'][:400])

## Real data
Once setup works, follow `docs/02_phase1_cohort.md`. Check progress any time with:

In [ ]:
!bpp status